In [1]:
import re

# ----Deterministic Approach ----
def deterministic_guardrail(text:str) -> bool:
    """ Returns True if content is blocked by guardrails, False otherwise. """
    banned_patterns = ["hack", "malware", "phishing", "exploit", "trojan", "virus"]
    return any(re.search(pattern, text, re.IGNORECASE) for pattern in banned_patterns)

test_inputs = [
    "how to hack a computer",
    "what is capital of france",
    "how to create malware",
]

print("Deterministic Guardrail Results:")
for input_text in test_inputs:
    result = deterministic_guardrail(input_text)
    status = "Blocked" if result else "Allowed"
    print(f"Input: '{input_text}' -> {status}")




Deterministic Guardrail Results:
Input: 'how to hack a computer' -> Blocked
Input: 'what is capital of france' -> Allowed
Input: 'how to create malware' -> Blocked


In [6]:
from langchain_anthropic import ChatAnthropic

# --- Model-based approach----
def model_based_guardrail(text : str) -> str:
    """Uses an LLM to evaluate content safety. Returns safe or unsafe"""
    model=ChatAnthropic(model="claude-sonnet-4-6", temperature=0.7)
    prompt=f"""Is the following user input safe to process? Reply with only 'SAFE' or 'UNSAFE' .
    Input:{text}
    """
    result=model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("\nModel-based Guardrail Results:")
for input_text in test_inputs:
    verdict=model_based_guardrail(input_text)
    status="UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{input_text}")



Model-based Guardrail Results:
UNSAFE:how to hack a computer
SAFE:what is capital of france
UNSAFE:how to create malware


In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query : str)->str:
    """Look up customer information."""
    return f"Customer record found for query:{query}"

# create agent with PII Middleware
agent=create_agent(
    model="claude-sonnet-4-6",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit crads in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ],
)

print("Agent with PII middleware created successfully.")

Agent with PII middleware created successfully.


In [10]:
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My email is john.deo@gmail.com and my card is 5747-4434-3432-5343 Can you help me?"
    }]
})

print("--Agent Response--")
print(result["messages"][-1].content)

--Agent Response--
I've located your account! However, you haven't told me what you actually need help with. 😊 What can I assist you with today?

Also, a friendly reminder: **never share your full credit card number** in chat, email, or any unsecured channel. If card verification is needed, legitimate systems will only ever ask for the **last 4 digits**. You may want to monitor your account for any suspicious activity.


In [5]:
# Test API KEY
try:
    result=agent.invoke({
        "messages":[{
            "role":"user",
            "content":"here is my key: sk-abcdffirhfrgneroghr8fkjdbdvjl"
        }]
    })

except Exception as e:
    print(f"Blocked as expected: {e}")

In [9]:
## Built -in Guardrails Human-in-the-loop Middleware

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_web(query:str)->str:
    """Search the web for information."""
    return f"Search results for query:{query}"

@tool
def send_email(to:str, subject:str,body:str)->str:
    """Send an email to a recipient"""
    return f"Email sent to {to} with subject:{subject}"

@tool
def delete_records(table: str,condition:str)->str:
    """delete records from the database"""
    return f"Deleted records from {table} where {condition}"


# Create agent with HTML Middleware
hitl_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

print("Agent with Human-in-the-loop middleware created successfully.")


Agent with Human-in-the-loop middleware created successfully.


In [11]:
config={"configurable":{"thread_id":"session_001"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Please delete all records from the database where status is 'inactive'.",
            }
        ]
    },
    config=config,
)

print("--Agent Response--")
print(result)

AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_01QVug7NtYBx6ZfGjNVxRvP1. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CgMBKbWF4vKMMsgoLXYgF'}

In [12]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for query: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    # Demo only: no actual email is sent.
    return f"Email prepared for {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete database records matching a condition."""
    # Demo only: no actual database records are deleted.
    return f"Deleted records from {table} where {condition}"


checkpointer = InMemorySaver()

hitl_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        )
    ],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "session_001"}}

# Step 1: Run the agent
result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Please delete all records from the database "
                    "where status is inactive."
                ),
            }
        ]
    },
    config=config,
)

print("Initial agent result:")
print(result)

# Step 2: Inspect the result.
# If the agent is interrupted, review the pending action
# before deciding whether to approve or reject it.
if "__interrupt__" in result:
    for interruption in result["__interrupt__"]:
        print("Pending approval:", interruption.value)

    # Step 3: Resume with an explicit approval.
    # Use only after reviewing the pending action.
    approved_result = hitl_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config,
    )

    print("Result after approval:")
    print(approved_result)


Initial agent result:
{'messages': [HumanMessage(content='Please delete all records from the database where status is inactive.', additional_kwargs={}, response_metadata={}, id='08953c01-bdd2-4991-8b30-c72d6a72210d'), AIMessage(content=[{'text': "I'll delete all inactive records from the database right away!", 'type': 'text'}, {'id': 'toolu_015Q9anWWNSTk5UfzdSgGqiG', 'caller': {'type': 'direct'}, 'input': {'table': 'records', 'condition': "status = 'inactive'"}, 'name': 'delete_records', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgMBXJGVtuMeTWNvWNF2x', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 716, 'output_tokens': 88, 'output_tokens_details': None, 'serv

In [23]:
## Before Agent Hook Input Filter
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keyboards.
    this runs Before the agent processes anything - zero llm cost for blocked requests"""

    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords=[kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:
        if not state["messages"]:
            return None

        first_message=state["messages"][0]
        if first_message.type != "human":
            return None

        content=first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked keyword detected: '{keyword}'")
                return {
                    "messages":[{
                        "role":"assistant",
                        "content":(
                            "I Can not process request containing inappropriate content."
                            "Please rephrase your request"
                        )
                    }],
                    "jump_to":"end"
                }
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")
        

Content filter agent created!


In [18]:

# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
## Machine Learning

**Machine learning (ML)** is a branch of **artificial intelligence (AI)** that enables computers to **learn from data and improve their performance over time without being explicitly programmed** for every task.

---

### 🔑 Core Idea
Instead of writing specific rules for a computer to follow, machine learning allows systems to:
1. **Analyze large amounts of data**
2. **Identify patterns and relationships**
3. **Make decisions or predictions** based on what they've learned

---

### 📚 Main Types of Machine Learning

| Type | Description | Example |
|------|-------------|---------|
| **Supervised Learning** | The model is trained on labeled data (input-output pairs) | Spam email detection |
| **Unsupervised Learning** | The model finds hidden patterns in unlabeled data | Customer segmentation |
| **Reinforcement Learning** | The model learns by trial and error, receiving rewards or penalties | Game-playing AI (e.g., Chess, Go) |
| **Semi-Supe

In [24]:
result = filtered_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How do I hack into a server?"
        }
    ]
})
print(" Unsafe request response:")
print(result["messages"][-1].content)

Blocked keyword detected: 'hack'
 Unsafe request response:
I Can not process request containing inappropriate content.Please rephrase your request


In [26]:
### Custom Guardrails with After Agent Hook(Output Safety)
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [27]:

# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
I'm sorry, but I don't have access to real-time weather data, and the available tool I have is a general-purpose tool that doesn't support live weather lookups. To find out today's weather, I'd recommend:

1. **Checking a weather website** like [weather.com](https://www.weather.com) or [Weather Underground](https://www.wunderground.com).
2. **Using a search engine** like Google and searching "weather today" — it will show your local forecast instantly.
3. **Checking a weather app** on your smartphone (like Weather, AccuWeather, etc.).
4. **Asking a voice assistant** like Siri, Alexa, or Google Assistant.

If you let me know your location, I can try to help in another way! 😊


Section 7: Layered / Combined Guardrails
Stack multiple guardrails in the middleware=[] array. They execute in order, building layered protection.

User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


Section 8: Real-World Use Case — Healthcare Chatbot
A healthcare chatbot that:

Blocks off-topic or harmful requests
Redacts patient PII (emails, credit card numbers)
Requires human approval before booking appointments
Validates that outputs are medically appropriate

In [29]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage

# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None


# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [30]:
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

result

{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='77593736-9c2b-42fd-b7df-e8b67293fee3'),
  AIMessage(content=[{'text': 'Sure! Let me look that up for you right away.', 'type': 'text'}, {'id': 'toolu_011NLgCHd64hDWQjPtB76Y6C', 'caller': {'type': 'direct'}, 'input': {'symptoms': 'Type 2 Diabetes'}, 'name': 'search_symptoms', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgMF4dtexNAuDwwBbzNAF', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 744, 'output_tokens': 71, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'diagnostics': None, 'm

In [31]:
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
Here are some common **over-the-counter (OTC) medications** for headaches:

---

### 💊 Common Headache Medications:

1. **Ibuprofen (Advil, Motrin)**
   - An anti-inflammatory (NSAID) that helps relieve pain and reduce inflammation.
   - Typically taken every 4–6 hours as needed.
   - ⚠️ Avoid on an empty stomach; not recommended for those with kidney issues or stomach ulcers.

2. **Acetaminophen (Tylenol)**
   - A pain reliever and fever reducer, gentler on the stomach.
   - A good option for those who cannot take NSAIDs.
   - ⚠️ Avoid exceeding the recommended dose, as high amounts can affect the liver.

3. **Aspirin (Bayer)**
   - An NSAID that relieves mild to moderate headache pain.
   - ⚠️ Not recommended for children or teenagers, and should be used cautiously by those on blood thinners.

---

### 💡 Non-Medication Tips:
- 💧 Stay **hydrated** — dehydration is a common headache trigger.
- 😴 Get adequate **rest and sleep**.
- 🧘 Try **relaxation techniques

In [32]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I'm sorry, but I'm **unable to assist with that request**. 🚫

Synthesizing drugs at home is:

- ⚠️ **Extremely Dangerous** – It can cause serious injury, chemical burns, explosions, or even death.
- ⚖️ **Illegal** – Manufacturing controlled substances is a serious criminal offense that can result in heavy fines and imprisonment.
- 🏥 **A Health Risk** – Improperly made substances can be toxic and life-threatening.

---

### ✅ What I *Can* Help You With:
As a healthcare assistant, I'm here to help you with:
- 🔍 **Searching for symptoms** and general health information.
- 💊 **Providing information about legitimate, approved medications**.
- 📅 **Booking medical appointments** with qualified healthcare professionals.

---

### 🆘 If You Need Help:
If you or someone you know is struggling with substance use, please reach out for support:
- **SAMHSA Helpline**: 1-800-662-4357 (free, confidential, 24/7)
- **Crisis Text Line**: Text HOME to 741741

A qualified healthcare 

In [33]:

# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)

=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='b2906128-a898-45b8-b87e-a2c118f05acf'), AIMessage(content="I'd be happy to help book an appointment with Dr. Sharma on March 15! I just need one more detail:\n\n- **What is your name?** (patient name required for the booking)\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*", additional_kwargs={}, response_metadata={'id': 'msg_011CgMFF94NR12b5Mb9eKmzm', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 746, 'output_tokens': 46, 'output_tokens_details': None, 'server_tool_